# Study2 Close Price Forecasting and Framework Validation

This notebook validates Brain-AI time-series backend availability and runs forecasting experiments using the last 1 year as test horizon.

It includes:
- Framework backend status and TimeSeriesAutoML execution
- Statistical and decomposition-based forecasting models
- Comprehensive forecast error metrics
- Forecast vs actual visualization for one stock
- Auto-generation of modality-specific notebook templates

In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make local source importable without requiring pip install -e .
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from brain_automl.config import get_default_config
from brain_automl.core import BACKEND_REGISTRY
from brain_automl.model_zoo.time_series_ai import TimeSeriesAutoML

warnings.filterwarnings("ignore")

DATA_PATH = Path.cwd() / "study2_volatility_dataset.csv"
print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Data exists: {DATA_PATH.exists()}")

/Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-03 13:27:18,651	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-04-03 13:27:18,773	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Project root: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai
Data path: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/study2_volatility_dataset.csv
Data exists: True


In [2]:
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

print(f"Shape: {df.shape}")
print("Columns:", list(df.columns))

DATE_COL = "Date"
TARGET_COL = "Close"
GROUP_COL = "Stock" if "Stock" in df.columns else None

if GROUP_COL:
    stock_counts = df[GROUP_COL].value_counts().head(10)
    print("Top stocks by row count:")
    display(stock_counts.to_frame("rows"))

display(df.head())

Shape: (18381, 12)
Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Return', 'MA_5', 'MA_10', 'MA_20', 'Volatility', 'Stock']
Top stocks by row count:


,rows
Stock,
HDFCBANK,3681
TCS,3681
RELIANCE,3681
BANKNIFTY,3677
NIFTY50,3661


,Date,Close,High,Low,Open,Volume,Return,MA_5,MA_10,MA_20,Volatility,Stock
0,2010-02-01,70.289337,71.530576,69.641253,71.530576,15034660,-0.019376,70.985294,73.677579,74.229110,0.000375,HDFCBANK
1,2010-02-02,69.382011,70.954982,69.070051,70.607870,20669400,-0.012992,70.303383,72.853526,73.950984,0.000169,HDFCBANK
2,2010-02-03,71.299896,71.723900,69.333688,70.300316,15455340,0.027267,70.527908,72.166122,73.765456,0.000744,HDFCBANK
3,2010-02-04,71.036263,71.394356,69.663210,71.091185,17210480,-0.003704,70.734415,71.553850,73.564221,0.000014,HDFCBANK
4,2010-02-05,69.129372,70.212434,68.762489,70.212434,18693880,-0.027211,70.227376,70.946192,73.257866,0.000740,HDFCBANK


In [3]:
# Use one stock for a univariate forecasting benchmark.
work_df = df.copy()
work_df[DATE_COL] = pd.to_datetime(work_df[DATE_COL], errors="coerce")
work_df = work_df.dropna(subset=[DATE_COL, TARGET_COL])

if GROUP_COL:
    selected_stock = work_df[GROUP_COL].iloc[0]
    work_df = work_df[work_df[GROUP_COL] == selected_stock].copy()
else:
    selected_stock = "single_series"

ts_df = work_df[[DATE_COL, TARGET_COL]].sort_values(DATE_COL).reset_index(drop=True)
ts_df = ts_df.rename(columns={DATE_COL: "ds", TARGET_COL: "y"})

TEST_HORIZON = min(252, max(30, len(ts_df) // 5))
train_df = ts_df.iloc[:-TEST_HORIZON].copy()
test_df = ts_df.iloc[-TEST_HORIZON:].copy()

print(f"Selected stock: {selected_stock}")
print(f"Total rows: {len(ts_df)}")
print(f"Train rows: {len(train_df)}")
print(f"Test rows (last ~1 year): {len(test_df)}")
print(f"Train range: {train_df['ds'].min().date()} -> {train_df['ds'].max().date()}")
print(f"Test range: {test_df['ds'].min().date()} -> {test_df['ds'].max().date()}")

Selected stock: HDFCBANK
Total rows: 3681
Train rows: 3429
Test rows (last ~1 year): 252
Train range: 2010-02-01 -> 2023-12-20
Test range: 2023-12-21 -> 2024-12-31


In [4]:
# Check configured framework backends and direct import availability.
config = get_default_config()
configured_backends = config["backends"]["by_modality"]["time_series"]["default"]

direct_import_checks = {
    "autogluon_timeseries": "autogluon.timeseries",
    "statsforecast": "statsforecast",
    "neuralforecast": "neuralforecast",
    "flaml": "flaml",
}

availability_rows = []
for b in configured_backends:
    registered = BACKEND_REGISTRY.has(b)
    backend_available = False
    if registered:
        backend_cls = BACKEND_REGISTRY.get(b)
        backend_available = bool(backend_cls.is_available())

    import_name = direct_import_checks.get(b, b)
    direct_import_available = False
    try:
        __import__(import_name)
        direct_import_available = True
    except Exception:
        direct_import_available = False

    availability_rows.append({
        "backend": b,
        "registered": registered,
        "framework_import_available": backend_available,
        "direct_import_available": direct_import_available,
    })

availability_df = pd.DataFrame(availability_rows)
display(availability_df)

excluded_backends = {"neuralforecast"}
execution_mask = (
    availability_df["registered"]
    & availability_df["framework_import_available"]
    & ~availability_df["backend"].isin(excluded_backends)
 )
available_backends = availability_df.loc[execution_mask, "backend"].tolist()

print("Excluded from notebook test run:", sorted(excluded_backends))
print("Available backends for framework execution:", available_backends)

,backend,registered,framework_import_available,direct_import_available
0,autogluon_timeseries,True,True,True
1,statsforecast,True,True,True
2,neuralforecast,True,True,True
3,flaml,True,True,True


Excluded from notebook test run: ['neuralforecast']
Available backends for framework execution: ['autogluon_timeseries', 'statsforecast', 'flaml']


In [5]:

# Run all available backends via the high-level forecast_last_horizon API.
# This handles standardisation, train/test split, per-backend execution, and
# metric computation internally — no manual reshaping needed.
run_config = get_default_config()
run_config["backends"]["hard_fail_if_no_backend_available_for_modality"] = False
run_config["modalities"]["allow_partial_success"] = True

executor = TimeSeriesAutoML(config=run_config)
framework_output = executor.forecast_last_horizon(
    data=df,
    timestamp_column=DATE_COL,
    target_column=TARGET_COL,
    item_id_column=GROUP_COL,
    horizon=TEST_HORIZON,
    backends=available_backends,
)

print(f"\nOutput dir : {framework_output['output_dir']}")
print(f"Item       : {framework_output['item_id']}")
print(f"Frequency  : {framework_output['frequency']}")
print(f"Horizon    : {framework_output['horizon']}")
print(f"Backends   : {[r.backend for r in framework_output['results']]}")
statuses = {r.backend: r.metadata.get('status', 'unknown') for r in framework_output['results']}
print(f"Statuses   : {statuses}")


[13:27:21] INFO     Brain-AI run started
[13:27:21] INFO     Log file : /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output/logs/brain_automl_20260403_132721.log
[13:27:21] INFO     Output dir: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output
[13:27:22] INFO     Item: HDFCBANK | Freq: B | Horizon: 252 | Train rows: 3640 | Backends: ['autogluon_timeseries', 'statsforecast', 'flaml']


Backends:   0%|          | 0/3 [00:00<?, ?backend/s]

[13:27:22] INFO     [START] autogluon_timeseries


[13:27:35] INFO     [ OK ] autogluon_timeseries — 13.1s | RMSE=67.9081  MAE=57.0559


Backends:  33%|███▎      | 1/3 [00:13<00:26, 13.07s/backend]

[13:27:35] INFO     [START] statsforecast
[13:27:35] INFO     [ OK ] statsforecast — 0.8s | RMSE=64.7201  MAE=53.3055


Backends:  67%|██████▋   | 2/3 [00:13<00:05,  5.86s/backend]

[13:27:35] INFO     [START] flaml
[13:28:26] INFO     [ OK ] flaml — 51.1s | RMSE=67.2496  MAE=57.1040


Backends: 100%|██████████| 3/3 [01:04<00:00, 21.65s/backend]

[13:28:26] INFO     Best backend: statsforecast | RMSE=64.7201  MAE=53.3055
[13:28:26] INFO     forecast_last_horizon complete

Output dir : /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output
Item       : HDFCBANK
Frequency  : B
Horizon    : 252
Backends   : ['autogluon_timeseries', 'statsforecast', 'flaml']
Statuses   : {'autogluon_timeseries': 'ok', 'statsforecast': 'ok', 'flaml': 'ok'}


In [6]:
# Display Brain-AI framework backend metrics table (sorted by RMSE).
fw_metrics = framework_output["metrics"]

if fw_metrics is not None and not fw_metrics.empty:
    display_cols = [c for c in ["backend", "mae", "rmse", "mape", "smape", "mase", "r2"] if c in fw_metrics.columns]
    print("=== Brain-AI Framework Backend Metrics ===")
    display(fw_metrics[display_cols].round(4))
else:
    print("No framework metrics produced — per-backend statuses:")
    for r in framework_output["results"]:
        status = r.metadata.get("status", "unknown")
        err = r.metadata.get("error", "")
        print(f"  {r.backend}: {status}  {'— ' + err[:150] if err else ''}")
    fw_metrics = None


=== Brain-AI Framework Backend Metrics ===


,backend,mae,rmse,mape,smape,mase,r2
0,statsforecast,53.3055,64.7201,7.1325,6.8078,6.3239,0.0716
1,flaml,57.1040,67.2496,7.3983,7.2641,6.7746,-0.0024
2,autogluon_timeseries,57.0559,67.9081,7.4111,7.2579,6.7689,-0.0221


In [7]:

# ── Predicted vs Actual — one chart per model, plus one combined overview ──
fw_preds = framework_output["predictions"]          # ds | actual | <backend>…
fw_metrics = framework_output["metrics"]            # may be None if all failed

backend_cols = [c for c in fw_preds.columns if c not in ("ds", "actual")]
n_models     = len(backend_cols)

if n_models == 0:
    print("No successful backend predictions to plot.")
else:
    COLORS = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
        "#9467bd", "#8c564b", "#e377c2", "#17becf",
    ]

    # ── 1. Individual per-model subplots (actual vs predicted) ──────────────
    rows = (n_models + 1) // 2          # 2 columns
    cols = min(n_models, 2)
    subplot_titles = [f"{b.replace('_', ' ').title()}" for b in backend_cols]

    fig_individual = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=subplot_titles,
        shared_xaxes=False,
        vertical_spacing=0.12,
        horizontal_spacing=0.08,
    )

    for idx, backend_name in enumerate(backend_cols):
        r = idx // 2 + 1
        c = idx % 2 + 1
        color = COLORS[idx % len(COLORS)]

        # Actual
        fig_individual.add_trace(
            go.Scatter(
                x=fw_preds["ds"], y=fw_preds["actual"],
                name="Actual" if idx == 0 else None,
                showlegend=(idx == 0),
                line=dict(color="black", width=1.5),
                legendgroup="actual",
            ),
            row=r, col=c,
        )
        # Predicted
        rmse_str = ""
        if fw_metrics is not None and not fw_metrics.empty:
            row_m = fw_metrics[fw_metrics["backend"] == backend_name]
            if not row_m.empty and "rmse" in row_m.columns:
                rmse_str = f"  RMSE={float(row_m['rmse'].iloc[0]):.2f}"

        fig_individual.add_trace(
            go.Scatter(
                x=fw_preds["ds"], y=fw_preds[backend_name],
                name=f"{backend_name}{rmse_str}",
                line=dict(color=color, width=1.5, dash="dash"),
                legendgroup=backend_name,
            ),
            row=r, col=c,
        )

    fig_individual.update_layout(
        title=dict(
            text=f"Predicted vs Actual — {selected_stock} — per model",
            font=dict(size=16),
        ),
        height=320 * rows,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified",
    )
    fig_individual.update_xaxes(title_text="Date")
    fig_individual.update_yaxes(title_text="Close Price")
    fig_individual.show()

    # ── 2. Combined overlay chart — all models on one axis ──────────────────
    fig_combined = go.Figure()

    # Training context (last 60 days of training data for context)
    context_df = framework_output["train_data"].tail(60)
    fig_combined.add_trace(go.Scatter(
        x=context_df["ds"], y=context_df["y"],
        name="Train (context)",
        line=dict(color="#aaaaaa", width=1, dash="dot"),
    ))

    # Actual test values
    fig_combined.add_trace(go.Scatter(
        x=fw_preds["ds"], y=fw_preds["actual"],
        name="Actual",
        line=dict(color="black", width=2),
    ))

    # Each backend's forecast
    for idx, backend_name in enumerate(backend_cols):
        color = COLORS[idx % len(COLORS)]
        rmse_str = ""
        if fw_metrics is not None and not fw_metrics.empty:
            row_m = fw_metrics[fw_metrics["backend"] == backend_name]
            if not row_m.empty and "rmse" in row_m.columns:
                rmse_str = f"  RMSE={float(row_m['rmse'].iloc[0]):.2f}"
        fig_combined.add_trace(go.Scatter(
            x=fw_preds["ds"], y=fw_preds[backend_name],
            name=f"{backend_name}{rmse_str}",
            line=dict(color=color, width=1.5, dash="dash"),
        ))

    # Shade the forecast region
    if len(fw_preds) > 0:
        split_date = fw_preds["ds"].iloc[0]
        fig_combined.add_vrect(
            x0=split_date, x1=fw_preds["ds"].iloc[-1],
            fillcolor="lightblue", opacity=0.08, line_width=0,
            annotation_text="Forecast window", annotation_position="top left",
        )

    fig_combined.update_layout(
        title=dict(
            text=f"All Models — Predicted vs Actual — {selected_stock}  (last {TEST_HORIZON} trading days)",
            font=dict(size=16),
        ),
        xaxis_title="Date",
        yaxis_title="Close Price",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=520,
        hovermode="x unified",
        template="plotly_white",
    )
    fig_combined.show()

    # ── 3. Metric comparison bar chart ───────────────────────────────────────
    if fw_metrics is not None and not fw_metrics.empty and "rmse" in fw_metrics.columns:
        metric_plot_df = fw_metrics[["backend", "rmse", "mae"]].dropna().sort_values("rmse")

        fig_bar = make_subplots(rows=1, cols=2, subplot_titles=["RMSE (lower is better)", "MAE (lower is better)"])
        for col_idx, metric in enumerate(["rmse", "mae"]):
            fig_bar.add_trace(
                go.Bar(
                    y=metric_plot_df["backend"],
                    x=metric_plot_df[metric],
                    orientation="h",
                    marker_color=COLORS[:len(metric_plot_df)],
                    text=metric_plot_df[metric].round(2),
                    textposition="outside",
                    name=metric.upper(),
                    showlegend=False,
                ),
                row=1, col=col_idx + 1,
            )

        fig_bar.update_layout(
            title=dict(text=f"Model Comparison — {selected_stock}", font=dict(size=16)),
            height=300,
            template="plotly_white",
        )
        fig_bar.show()
